# Fib (Python)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/Fib_python.ipynb)

Chapter 14 — Recursion. Runnable notebook of `Fib`: the algorithm in Python, run on the examples of the chapter, with the «Try it» cell to run it on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

The same notebook in [C++](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/Fib_cpp.ipynb) and [Java](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/Fib_java.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion_python.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. Python 3, nothing to install.

**Names.** There are no arrays in this chapter: variables and parameters have the names of the pseudo-code. Two exceptions in Python, where `from` and `next` have a special meaning: `Hanoi` uses `frm` instead of `from` and `IterativeFib` uses `nxt` instead of `next`.

## Tools: counters, stack and trace

The cell defines the counters (`cnt`, `calls`, `max_depth`), the call stack (`push` and `pop`) and the functions that print the trace (`event`, `say`) and the table of steps (`print_steps`). It has to be run only once.

In [1]:
cnt = [0] * 20      # cnt[r] = number of executions of line r (lines numbered as in the lecture notes)
calls = 0           # number of calls = nodes of the tree of recursive calls
max_depth = 0       # depth of the recursion = maximum height of the stack
mults = 0           # multiplications (FastPower)
moves = 0           # moves (Hanoi)
ev = 0              # counter of the events of the trace
stack = []          # the call stack: the arguments of the active calls
seen = [0] * 64     # seen[x] = number of calls with argument x
trace = False       # True: print the trace of the execution


def reset():
    global calls, max_depth, mults, moves, ev
    calls = max_depth = mults = moves = ev = 0
    stack.clear()
    for r in range(len(cnt)):
        cnt[r] = 0
    for k in range(len(seen)):
        seen[k] = 0


def push(x):                       # a call starts: its frame goes on top of the stack
    global calls, max_depth
    calls += 1
    stack.append(x)
    max_depth = max(max_depth, len(stack))
    if x < len(seen):
        seen[x] += 1


def pop():                         # the call returns: its frame leaves the stack
    stack.pop()


def event(text):                   # numbered line of the trace, with the stack
    global ev
    ev += 1
    print("%2d. %s   stack: %s" % (ev, text, stack))


def say(call, action):             # line of the trace: the call, indented one level for every active call
    print("%-28s| %s" % ("  " * (len(stack) - 1) + call, action))


def print_steps(*lines):
    print("Line | Executions")
    for r in lines:
        print("%4d | %10d" % (r, cnt[r]))
    print("Total steps S = %d" % sum(cnt))

## Fibonacci numbers: Fib

**Input:** a natural number $n$. **Output:** $F(n)$, with $F(0) = 0$, $F(1) = 1$ and $F(n) = F(n-1) + F(n-2)$. Naive recursive algorithm: $C(n) = 2F(n+1) - 1$ calls and $S(n) = 4F(n+1) - 2 = 2^{\Theta(n)}$ steps.

Pseudo-code: section 14.5 of the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

In [2]:
def fib(n):
    push(n); cnt[2] += 1
    if n <= 1:
        cnt[3] += 1
        if trace: say("Fib(%d)" % n, "base case, returns %d" % n)
        pop()
        return n
    if trace: say("Fib(%d)" % n, "calls Fib(%d)" % (n - 1))
    x = fib(n - 1)                                 # the two calls are separated only to print the trace
    if trace: say("Fib(%d)" % n, "calls Fib(%d)" % (n - 2))
    y = fib(n - 2)
    cnt[5] += 1
    if trace: say("Fib(%d)" % n, "returns %d + %d = %d" % (x, y, x + y))
    pop()
    return x + y

In [3]:
def fib_value(k):                                  # F(k) with a loop, to check
    a, b = 0, 1
    for _ in range(k):
        a, b = b, a + b
    return a


# Example of the lecture notes: the execution of Fib(4), in depth-first order
n = 4
reset()
trace = True
print("%-28s| %s" % ("call", "action"))
r = fib(n)
trace = False
print("F(%d) = %d, calls C(%d) = %d = 2F(%d) - 1, depth = %d" % (n, r, n, calls, n + 1, max_depth))

# Fib(5): the same subproblems are solved many times
print()
n = 5
reset()
r = fib(n)
print("Fib(%d) = %d: %d calls, depth %d, S = %d steps" % (n, r, calls, max_depth, sum(cnt)))
print("  k | calls of Fib(k)")
for k in range(n, -1, -1):
    print("%3d | %d" % (k, seen[k]))
print_steps(2, 3, 5)

# Number of calls and steps: C(n) = 2F(n+1) - 1 and S(n) = 4F(n+1) - 2
print()
print("  n |   F(n) |     C(n) | 2F(n+1) - 1 |     S(n) | 4F(n+1) - 2 | depth")
for n in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 20, 30]:
    reset()
    r = fib(n)
    print("%3d | %6d | %8d | %11d | %8d | %11d | %d" % (n, r, calls, 2 * fib_value(n + 1) - 1, sum(cnt), 4 * fib_value(n + 1) - 2, max_depth))

call                        | action
Fib(4)                      | calls Fib(3)
  Fib(3)                    | calls Fib(2)
    Fib(2)                  | calls Fib(1)
      Fib(1)                | base case, returns 1
    Fib(2)                  | calls Fib(0)
      Fib(0)                | base case, returns 0
    Fib(2)                  | returns 1 + 0 = 1
  Fib(3)                    | calls Fib(1)
    Fib(1)                  | base case, returns 1
  Fib(3)                    | returns 1 + 1 = 2
Fib(4)                      | calls Fib(2)
  Fib(2)                    | calls Fib(1)
    Fib(1)                  | base case, returns 1
  Fib(2)                    | calls Fib(0)
    Fib(0)                  | base case, returns 0
  Fib(2)                    | returns 1 + 0 = 1
Fib(4)                      | returns 2 + 1 = 3
F(4) = 3, calls C(4) = 9 = 2F(5) - 1, depth = 4

Fib(5) = 5: 15 calls, depth 5, S = 30 steps
  k | calls of Fib(k)
  5 | 1
  4 | 1
  3 | 2
  2 | 3
  1 | 5
  0 | 3
Line | Ex

**Try it yourself.** Change $n$ and see how fast the calls grow.

In [4]:
n = 6        # <-- change n (with n > 35 it takes a while)
reset()
r = fib(n)
print("F(%d) = %d, calls = %d, S = %d" % (n, r, calls, sum(cnt)))

F(6) = 8, calls = 25, S = 50


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).